# Comprehensive Mobile Price Analysis

This notebook covers the full workflow for mobile price classification:
1. Data loading and exploration
2. Data cleaning and preprocessing
3. Statistical analysis with NumPy and SciPy
4. Data visualization with Matplotlib
5. Insight synthesis and conclusion

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
plt.style.use('ggplot')

## 1) Data Loading and Exploration

In [ ]:
local_path = 'train.csv'
fallback_url = 'https://raw.githubusercontent.com/devtlv/MiniProject-DataAnalysis-W6D5-Mobile_Price_Classification/main/train.csv'

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
    source_used = local_path
else:
    df = pd.read_csv(fallback_url)
    source_used = fallback_url

print(f'Dataset loaded from: {source_used}')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Column types:')
display(df.dtypes.to_frame('dtype'))

print('Basic descriptive statistics:')
display(df.describe(include='all').transpose())

print('Target distribution (price_range):')
display(df['price_range'].value_counts().sort_index().to_frame('count'))

## 2) Data Cleaning and Preprocessing

In [ ]:
missing_counts = df.isnull().sum()
print('Missing values per column:')
display(missing_counts[missing_counts > 0].to_frame('missing_count'))

# Fill missing numeric values with median if needed.
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Encode categorical columns only if present.
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f'Categorical columns encoded: {cat_cols if cat_cols else 
}')
print(f'Final shape after preprocessing: {df.shape}')

## 3) Statistical Analysis with NumPy and SciPy

In [ ]:
target_col = 'price_range'
feature_cols = [c for c in df.columns if c != target_col]

summary_rows = []
for col in feature_cols:
    arr = df[col].to_numpy()
    col_mode = stats.mode(arr, keepdims=False).mode
    summary_rows.append({
        'feature': col,
        'mean': np.mean(arr),
        'median': np.median(arr),
        'mode': col_mode,
        'range': np.ptp(arr),
        'variance': np.var(arr, ddof=1),
        'std_dev': np.std(arr, ddof=1),
        'skewness': stats.skew(arr, bias=False),
        'kurtosis': stats.kurtosis(arr, fisher=True, bias=False)
    })

stats_df = pd.DataFrame(summary_rows).set_index('feature').sort_index()
display(stats_df.round(4))

In [ ]:
anova_rows = []
for col in feature_cols:
    groups = [df.loc[df[target_col] == g, col].to_numpy() for g in sorted(df[target_col].unique())]

    f_stat, p_anova = stats.f_oneway(*groups)
    h_stat, p_kruskal = stats.kruskal(*groups)

    anova_rows.append({
        'feature': col,
        'anova_p_value': p_anova,
        'kruskal_p_value': p_kruskal,
        'anova_significant_0_05': p_anova < 0.05,
        'kruskal_significant_0_05': p_kruskal < 0.05
    })

hypothesis_df = pd.DataFrame(anova_rows).set_index('feature').sort_values('anova_p_value')
display(hypothesis_df)

In [ ]:
corr_rows = []
for col in feature_cols:
    pearson_r, pearson_p = stats.pearsonr(df[col], df[target_col])
    spearman_rho, spearman_p = stats.spearmanr(df[col], df[target_col])

    normaltest_p = np.nan
    sample_for_normaltest = df[col].to_numpy()
    if len(sample_for_normaltest) >= 8:
        _, normaltest_p = stats.normaltest(sample_for_normaltest)

    corr_rows.append({
        'feature': col,
        'pearson_r': pearson_r,
        'pearson_p': pearson_p,
        'spearman_rho': spearman_rho,
        'spearman_p': spearman_p,
        'normaltest_p': normaltest_p
    })

corr_df = pd.DataFrame(corr_rows).set_index('feature')
corr_df = corr_df.reindex(corr_df['spearman_rho'].abs().sort_values(ascending=False).index)
display(corr_df.round(4))

## 4) Data Visualization with Matplotlib

In [ ]:
# Histograms
n_features = len(feature_cols)
n_cols = 4
n_rows = int(np.ceil(n_features / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=20, color='teal', edgecolor='black', alpha=0.75)
    axes[i].set_title(f'Histogram: {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots for top correlated features vs target
top_features = corr_df.index[:4].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(top_features):
    axes[i].scatter(df[col], df[target_col], alpha=0.35, color='darkcyan')
    axes[i].set_title(f'{col} vs {target_col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel(target_col)

plt.tight_layout()
plt.show()

In [ ]:
# Box plots by price range for top features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(top_features):
    grouped = [df.loc[df[target_col] == grp, col].values for grp in sorted(df[target_col].unique())]
    axes[i].boxplot(grouped, labels=sorted(df[target_col].unique()))
    axes[i].set_title(f'Box plot of {col} by {target_col}')
    axes[i].set_xlabel(target_col)
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap using Matplotlib
corr_matrix = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)

ax.set_xticks(np.arange(len(corr_matrix.columns)))
ax.set_yticks(np.arange(len(corr_matrix.index)))
ax.set_xticklabels(corr_matrix.columns, rotation=90)
ax.set_yticklabels(corr_matrix.index)
ax.set_title('Feature Correlation Heatmap')

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Correlation Coefficient')

plt.tight_layout()
plt.show()

## 5) Insight Synthesis and Conclusion

In [ ]:
significant_anova = hypothesis_df[hypothesis_df['anova_significant_0_05']].index.tolist()
top_positive = corr_df.sort_values('spearman_rho', ascending=False).head(5)
top_negative = corr_df.sort_values('spearman_rho', ascending=True).head(5)

print('Key findings:')
print(f'- Number of statistically significant features (ANOVA, p<0.05): {len(significant_anova)}')
print('- Top positive Spearman correlations with price_range:')
display(top_positive[['spearman_rho', 'spearman_p']].round(4))

print('- Top negative Spearman correlations with price_range:')
display(top_negative[['spearman_rho', 'spearman_p']].round(4))

print('Interpretation guide:')
print('- Features with high absolute correlation and significant group differences are strong price determinants.')
print('- Features with weak correlation but significant p-values may have non-linear or segmented effects.')
print('- Use these signals for downstream feature selection and model design.')

### Final Conclusion

This analysis combines descriptive statistics, distribution diagnostics, hypothesis testing, correlation analysis, and visual exploration.
The strongest determinants of mobile price class are the features that consistently show:
- significant differences across price groups, and
- high absolute correlation with `price_range`.

Unexpected findings should be validated through additional checks (outlier handling, stratified analysis, and model-based feature importance).
Overall, this notebook provides a complete and reproducible analytical pipeline for the mobile price classification problem.